# SZGC: обучение модели сжатия на больших корпусах (RU + EN)

Собирает многостилевый двуязычный корпус (субтитры RU+EN, Википедия RU+EN, проза Taiga,
новости/техника OPUS в обе стороны) и обучает модель `graph-model.bin`:
словарь 600К словоформ, леммы, order-0/1/2, лемматическое согласование, LZP.

Далее ноутбук сам верифицирует модель: round-trip, детерминизм, бенчмарк против
gzip/bzip2/xz/brotli/zstd. Стадии можно перезапускать по отдельности.

Ручки масштаба: `PREFIX_B`, `cap` (вики), `TOTAL_CAP`, `DICT_CAP`, `CTX_B1_ROWS`,
`B2_ROWS`, `O0_TOP`, `*_KEEP` (в build_model.py).

In [ ]:
!pip -q install pymorphy3 numpy requests brotli zstandard
import sys, os
os.makedirs('/content/szgc', exist_ok=True)


In [ ]:
# ---------- 1. Загрузка корпусов: все стили, русский + английский ----------
import os, subprocess, zipfile
D = '/content/data'
os.makedirs(D + '/corpus', exist_ok=True)
os.makedirs(D + '/opus', exist_ok=True)

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True)

# OPUS: новости / веб-новости / техника (moses en-ru, берём ОБЕ стороны)
for name, url in [
    ("nc",     "https://object.pouta.csc.fi/OPUS-News-Commentary/v16/moses/en-ru.txt.zip"),
    ("gv",     "https://object.pouta.csc.fi/OPUS-GlobalVoices/v2018q4/moses/en-ru.txt.zip"),
    ("kde4",   "https://object.pouta.csc.fi/OPUS-KDE4/v2/moses/en-ru.txt.zip"),
    ("ubuntu", "https://object.pouta.csc.fi/OPUS-Ubuntu/v14.10/moses/en-ru.txt.zip"),
]:
    z = D + '/opus/%s.zip' % name
    if not os.path.exists(z):
        sh('curl -sL --retry 3 -o %s "%s"' % (z, url))
    with zipfile.ZipFile(z) as zf:
        for side, out_name in (('.ru', 'opus_%s.txt' % name), ('.en', 'opus_en_%s.txt' % name)):
            fn = [n for n in zf.namelist() if n.endswith(side)][0]
            lines = zf.read(fn).decode('utf-8', 'replace').split('\n')
            lines = [l.strip() for l in lines if l.strip()]
            with open(D + '/corpus/' + out_name, 'w') as f:
                f.write('\n'.join(' '.join(lines[i:i+12]) for i in range(0, len(lines), 12)))
    print('opus_%s: ok' % name)

# Разговорная речь: префиксы монокорпуса OpenSubtitles (RU + EN).
# Гигабайты по желанию: увеличьте PREFIX_B; частичная распаковка gzip корректна.
import zlib
PREFIX_B = 300 * 1024 * 1024   # 300 МБ сжатых ~= 1 ГБ текста на язык
for lang, cap in (("ru", 700 * 1024 * 1024), ("en", 700 * 1024 * 1024)):
    gz = subprocess.run(['curl', '-sL', '--retry', '3', '-r', '0-%d' % (PREFIX_B - 1),
                         'https://object.pouta.csc.fi/OPUS-OpenSubtitles/v2024/mono/%s.txt.gz' % lang],
                        capture_output=True).stdout
    d = zlib.decompressobj(16 + zlib.MAX_WBITS)
    out = d.decompress(gz)
    with open(D + '/corpus/subtitles_%s.txt' % lang, 'wb') as f:
        f.write(out[:cap])
    print('subtitles_%s: %d МБ' % (lang, min(len(out), cap) // 1048576))
    del gz, out

# Энциклопедия: шарды Википедии ru и en (parquet) через зеркало
import pyarrow.parquet as pq
def wiki(lang, shard, total, cap, exclude=()):
    sh('curl -sL --retry 3 -o /content/wiki_%s.parquet '
       '"https://hf-mirror.com/datasets/wikimedia/wikipedia/resolve/main/20231101.%s/%s"' % (lang, lang, shard))
    pf = pq.ParquetFile('/content/wiki_%s.parquet' % lang)
    sz = 0
    fbuf, fidx = [], 0
    def flush(fbuf, fidx):
        if fbuf:
            with open(D + '/corpus/wiki_%s_%02d.txt' % (lang, fidx), 'w') as f:
                f.write('\n\n'.join(fbuf))
        return [], fidx + 1
    for batch in pf.iter_batches(batch_size=2000, columns=['title', 'text']):
        for t, x in zip(batch.column('title').to_pylist(), batch.column('text').to_pylist()):
            if t.strip().lower().replace('ё', 'е') in exclude or len(x) < 1200 or sz >= cap:
                continue
            fbuf.append('@@@ %s @@@\n\n%s' % (t, x))
            sz += len(x)
            if sum(len(b) for b in fbuf) > 24_000_000:
                fbuf, fidx = flush(fbuf, fidx)
    fbuf, _ = flush(fbuf, fidx)
    print('wiki_%s: %.0f МБ' % (lang, sz / 1048576))

wiki('ru', 'train-00001-of-00021.parquet', 21, 300_000_000,
     exclude={'лев толстой', 'москва', 'интернет', 'солнечная система', 'фотосинтез',
              'дезоксирибонуклеиновая кислота', 'анна каренина', 'руслан и людмила'})
wiki('en', 'train-00000-of-00041.parquet', 41, 300_000_000)

# Художественная проза: Taiga proza (parquet, hf-mirror)
sh('curl -sL --retry 3 -o /content/taiga0.parquet '
   '"https://hf-mirror.com/datasets/cointegrated/taiga_stripped_proza/resolve/main/data/train-00000-of-00083-5a836a36820bbc21.parquet"')
pf = pq.ParquetFile('/content/taiga0.parquet')
sz = 0
with open(D + '/corpus/taiga_proza.txt', 'w') as f:
    for batch in pf.iter_batches(batch_size=500, columns=['text']):
        for t in batch.column('text').to_pylist():
            t = t.strip()
            if len(t) < 400:
                continue
            f.write(t + '\n\n')
            sz += len(t) + 2
            if sz >= 200_000_000:
                break
        if sz >= 200_000_000:
            break
print('taiga: %.0f МБ' % (sz / 1048576))

# Частотные списки OpenSubtitles (словарное покрытие RU + EN)
sh('curl -sL -o /content/ru_50k.txt "https://raw.githubusercontent.com/hermitdave/FrequencyWords/master/content/2018/ru/ru_50k.txt"')
sh('curl -sL -o /content/en_50k.txt "https://raw.githubusercontent.com/hermitdave/FrequencyWords/master/content/2018/en/en_50k.txt"')

total = sum(os.path.getsize(os.path.join(D, 'corpus', f)) for f in os.listdir(D + '/corpus'))
print('готово: корпус %.2f ГБ' % (total / 1e9))


In [ ]:
CORE_SRC = r'''
# -*- coding: utf-8 -*-
"""SZGC ядро: range-кодер, частотные таблицы base+delta, загрузчик модели SZGM.

Состояние: адаптация в течение одного вызова encode/decode накапливается в
delta-слое затронутых таблиц; откат = сброс delta/esc к base (touched-set).
encode и decode всегда стартуют с чистого снимка модели.
"""
import os
import re

M32 = 0xFFFFFFFF
TOP = 1 << 24
ADAPT = 24
RESIZE = 32768
CTX_CAP = 192
ESC_RANK = -1
ESC_LIT = -2
O0_SPLIT = int(os.environ.get("SZGC_O0_SPLIT", "32"))   # контекстный буст языка (0=выкл)
W_MATCH, ESC_MATCH = 4096, 384
PAIR_SHIFT = 21
W_LZP_HIT, W_LZP_MISS = 4096, 128      # флаг попытки LZP (адаптируется)
B_ESC = 12                              # стартовый вес escape байтовых строк
B_ASTEP = 48                            # шаг адаптации байтовых строк
W1_O0, W1_ROW, W1_FORCE = 64, 192, 48    # ternary-флаг order-1 (адаптируется)
MODEL_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)),
                          "graph-model.bin")

ABSENT = object()

_LAT = frozenset("abcdefghijklmnopqrstuvwxyz")
_CYR = frozenset("абвгдеёжзийклмнопрстуфхцчшщъыьэюя")


def word_lang(w):
    """0=смешанный/прочий, 1=кириллица, 2=латиница (по алфавиту словоформы)."""
    if not w:
        return 0
    s = set(w)
    if s <= _LAT:
        return 2
    if s <= _CYR:
        return 1
    return 0


def apply_case(w, f):
    if f == 0:
        return w
    if f == 1:
        return w[:1].upper() + w[1:]
    return w.upper()


# ------------------------------------------------------------- токенизация
WORD_RE = re.compile(r"[А-Яа-яЁёA-Za-z]+")


def tokenize(text):
    """Единственный токенизатор (зеркалится во всех реализациях кодека):
    чередование word/lit до конца строки."""
    out, pos = [], 0
    for m in WORD_RE.finditer(text):
        if m.start() > pos:
            out.append(("lit", text[pos:m.start()]))
        out.append(("word", m.group()))
        pos = m.end()
    if pos < len(text):
        out.append(("lit", text[pos:]))
    return out


def get_case(w):
    if w.islower():
        return 0
    if w[0].isupper() and (len(w) == 1 or w[1:].islower()):
        return 1
    if len(w) >= 2 and w.isupper():
        return 2
    return 3


# ------------------------------------------------------------- varint io
def put_varint(buf, n):
    v = n & 0xFFFFFFFFFFFFFFFF
    while True:
        b = v & 0x7F
        v >>= 7
        if v:
            buf.append(b | 0x80)
        else:
            buf.append(b)
            return


class Reader:
    def __init__(self, data):
        self.d = data
        self.p = 0

    def varint(self):
        v, sh = 0, 0
        while True:
            b = self.d[self.p]
            self.p += 1
            v |= (b & 0x7F) << sh
            sh += 7
            if not (b & 0x80):
                return v

    def blob(self):
        n = self.varint()
        b = self.d[self.p:self.p + n]
        self.p += n
        return b


# ------------------------------------------------------------ range coder
class RangeEncoder:
    """LZMA-стиль: кэш-байт + учёт переносов."""

    def __init__(self):
        self.low = 0
        self.range = 0xFFFFFFFF
        self.cache = 0
        self.cache_size = 1
        self.out = bytearray()

    def _shift_low(self):
        if (self.low & M32) < 0xFF000000 or (self.low >> 32) != 0:
            carry = (self.low >> 32) & 0xFF
            temp = self.cache
            while True:
                self.out.append((temp + carry) & 0xFF)
                temp = 0xFF
                self.cache_size -= 1
                if self.cache_size == 0:
                    break
            self.cache = (self.low >> 24) & 0xFF
        self.cache_size += 1
        self.low = (self.low << 8) & M32

    def encode(self, cum, freq, tot):
        assert freq > 0 and tot > 0 and cum + freq <= tot, (cum, freq, tot)
        r = self.range // tot
        self.low += cum * r
        self.range = freq * r
        while self.range < TOP:
            self._shift_low()
            self.range <<= 8

    def flush(self):
        for _ in range(5):
            self._shift_low()
        assert not self.out or self.out[0] == 0
        return bytes(self.out[1:])


class RangeDecoder:
    """PPMd/7-zip стиль: без low, нормализация вычитанием."""

    def __init__(self, data):
        self.d = data
        self.p = 0
        self.range = 0xFFFFFFFF
        self.code = 0
        for _ in range(4):
            self.code = ((self.code << 8) | self._byte()) & M32
        self.clamped = 0

    def _byte(self):
        b = self.d[self.p] if self.p < len(self.d) else 0
        self.p += 1
        return b

    def start(self, tot):
        self._r = self.range // tot
        cum = self.code // self._r
        if cum >= tot:
            cum = tot - 1
            self.clamped += 1
        return cum

    def finish(self, cum, freq):
        self.code = (self.code - cum * self._r) & M32
        self.range = freq * self._r
        while self.range < TOP:
            self.range <<= 8
            self.code = ((self.code << 8) | self._byte()) & M32


# --------------------------------------------------- таблица частот base+delta
class FT:
    """Частотная таблица: неизменяемая base + адаптивный delta-слой.

    static (o0): веса символов не адаптируются, esc-псевдосимволы живут
    в delta и восстанавливаются при reset().
    astep: шаг адаптации (по умолчанию ADAPT); cap: жёсткий лимит символов
    (для чисто адаптивных байтовых строк нужен 256 вместо формулы CTX_CAP)."""

    __slots__ = ("bs", "bw", "bset", "delta", "esc", "base_esc",
                 "base_sum", "is_o0", "esc0", "_tot", "astep", "cap", "resize_at")

    def __init__(self, syms=None, weights=None, esc=40, is_o0=False, esc0=(0, 0),
                 astep=None, cap=None, resize_at=None):
        self.bs = list(syms) if syms else []
        self.bw = list(weights) if weights else []
        self.bset = set(self.bs)
        self.esc = esc
        self.base_esc = esc
        self.base_sum = sum(self.bw)
        self.is_o0 = is_o0
        self.esc0 = esc0
        self.astep = ADAPT if astep is None else astep
        self.cap = cap
        self.resize_at = RESIZE if resize_at is None else resize_at
        if is_o0:
            self.delta = {ESC_RANK: esc0[0], ESC_LIT: esc0[1]}
            self._tot = self.base_sum + esc0[0] + esc0[1]
        else:
            self.delta = {}
            self._tot = self.base_sum + esc

    def _cap(self):
        return self.cap if self.cap is not None else CTX_CAP + self.base_sum // 16

    def total(self):
        return self._tot

    def total_excl(self, excl):
        """Сумма весов без исключённых символов (для dec.start зеркально enc)."""
        if not excl:
            return self._tot
        lo = 0
        delta = self.delta
        for i in range(len(self.bs)):
            s = self.bs[i]
            if s in excl:
                continue
            lo += self.bw[i] + delta.get(s, 0)
        for s, dw in delta.items():
            if s in self.bset or s in excl:
                continue
            lo += dw
        return lo + self.esc

    def reset(self):
        if self.is_o0:
            self.delta = {ESC_RANK: self.esc0[0], ESC_LIT: self.esc0[1]}
            self.esc = 0
            self._tot = self.base_sum + self.esc0[0] + self.esc0[1]
        else:
            self.delta = {}
            self.esc = self.base_esc
            self._tot = self.base_sum + self.esc

    def in_table(self, sym, excl=None):
        """Есть ли символ в таблице (и не исключён ли он)."""
        if sym in self.bset:
            return not (excl and sym in excl)
        if sym in self.delta and not (excl and sym in excl):
            return True
        return False

    def find_encode(self, sym, excl=None, ctx=0):
        """(cum, freq, tot) с учётом исключений; ESC-интервал, если sym нет."""
        lo = 0
        found = False
        freq = 0
        cum = 0
        delta = self.delta
        bs = self.bs
        bw = self.bw
        for i in range(len(bs)):
            s = bs[i]
            if excl is not None and s in excl:
                continue
            w = bw[i] + delta.get(s, 0)
            if w and s == sym:
                found = True
                freq = w
                cum = lo
            lo += w
        for s, dw in delta.items():
            if s in self.bset or (excl is not None and s in excl):
                continue
            if s == sym:
                found = True
                freq = dw
                cum = lo
            lo += dw
        tot = lo + self.esc
        if found:
            return cum, freq, tot
        return tot - self.esc, self.esc, tot

    def decode_sym(self, cum, excl=None):
        """(sym|None, lo, freq); None = escape."""
        lo = 0
        delta = self.delta
        bs = self.bs
        bw = self.bw
        for i in range(len(bs)):
            s = bs[i]
            if excl is not None and s in excl:
                continue
            w = bw[i] + delta.get(s, 0)
            if cum < lo + w:
                return s, lo, w
            lo += w
        for s, dw in delta.items():
            if s in self.bset or (excl is not None and s in excl):
                continue
            if cum < lo + dw:
                return s, lo, dw
            lo += dw
        return None, lo, self.esc

    def adapt(self, sym, insert=True, insert_sym=None):
        """sym=None -> escape (esc++ и, при insert, вставка insert_sym).
        sym=s -> вес символа +ADAPT (или вставка). Возвращает self."""
        if sym is None:
            if self.esc < 60000:
                self.esc += self.astep
                self._tot += self.astep
            if insert and insert_sym is not None:
                if insert_sym in self.delta:
                    if self.delta[insert_sym] < 60000:
                        self.delta[insert_sym] += self.astep
                        self._tot += self.astep
                elif insert_sym not in self.bset and \
                        len(self.bs) + len(self.delta) < self._cap():
                    self.delta[insert_sym] = self.astep
                    self._tot += self.astep
        elif sym in self.bset:
            old = self.delta.get(sym, 0)
            if old < 60000:
                self.delta[sym] = old + self.astep
                self._tot += self.astep
        else:
            old = self.delta.get(sym)
            if old is None:
                if len(self.bs) + len(self.delta) < self._cap():
                    self.delta[sym] = self.astep
                    self._tot += self.astep
            elif old < 60000:
                self.delta[sym] = old + self.astep
                self._tot += self.astep
        if self._tot >= self.resize_at and not self.is_o0:
            self._resize()
        return self

    def _resize(self):
        nd = {}
        bs = self.bs
        bw = self.bw
        delta = self.delta
        for i in range(len(bs)):
            s = bs[i]
            eff = (bw[i] + delta.get(s, 0)) >> 1
            d = eff - bw[i]
            if d > 0:
                nd[s] = d
        for s, dw in delta.items():
            if s in self.bset:
                continue
            eff = dw >> 1
            if eff > 0:
                nd[s] = eff
        self.delta = nd
        if self.is_o0:
            nd.setdefault(ESC_RANK, 0)
            nd[ESC_RANK] = max(1, nd[ESC_RANK])
            nd[ESC_LIT] = max(1, self.delta.get(ESC_LIT, 0) >> 1)
        self.esc = max(1, self.esc >> 1) if not self.is_o0 else self.esc
        self._tot = self.base_sum + sum(nd.values()) + self.esc


# ------------------------------------------------------------- модель
class FenwickO0:
    """Order-0 таблица (тысячи символов) на дереве Фенвика.

    API совместим с FT для ролей, нужных кодеку: find_encode/decode_sym/
    total/total_excl/in_table/adapt/reset + bset. ESC_RANK/ESC_LIT живут
    вне дерева (два хвостовых интервала).

    Контекстное взвешивание (split): если язык кодируемого символа совпадает
    с языком предыдущего слова (ctx=1 кириллица / 2 латиница / 0 прочее),
    вес символа умножается на split. ESC-интервалы масштабируются так, чтобы
    их вероятность не зависела от контекста. Два доп. дерева Фенвика по
    языкам дают префиксные суммы с множителем за O(log n). Только целые
    числа — байт-в-байт во всех реализациях."""

    __slots__ = ("bs", "bweight", "pos", "bset", "tree", "raw_tot",
                 "esc_rank", "esc_lit", "esc_rank0", "esc_lit0",
                 "extra", "adds", "delta", "lang", "split",
                 "tree_cyr", "tree_lat")

    def __init__(self, syms, weights, esc_rank=0, esc_lit=0, langs=None, split=0):
        self.bs = list(syms)
        self.bweight = list(weights)
        self.pos = {s: i for i, s in enumerate(self.bs)}
        self.bset = set(self.bs)
        n = len(self.bs)
        self.tree = [0] * (n + 1)
        for i, w in enumerate(self.bweight):
            self._tadd(self.tree, i + 1, w)
        self.raw_tot = sum(self.bweight)
        self.esc_rank = esc_rank
        self.esc_lit = esc_lit
        self.esc_rank0 = esc_rank
        self.esc_lit0 = esc_lit
        self.extra = {}
        self.adds = []
        self.delta = {}          # совместимость (не используется)
        self.lang = list(langs) if langs else [0] * n
        self.split = split if (split and len(self.lang) == n) else 0
        self.tree_cyr = [0] * (n + 1)
        self.tree_lat = [0] * (n + 1)
        if self.split:
            for i, lg in enumerate(self.lang):
                if lg == 1:
                    self._tadd(self.tree_cyr, i + 1, self.bweight[i])
                elif lg == 2:
                    self._tadd(self.tree_lat, i + 1, self.bweight[i])

    @staticmethod
    def _tadd(tree, i, v):
        n = len(tree) - 1
        while i <= n:
            tree[i] += v
            i += i & (-i)

    @staticmethod
    def _tpref(tree, i):
        """Сумма весов позиций 1..i (i — 1-based)."""
        s = 0
        while i > 0:
            s += tree[i]
            i -= i & (-i)
        return s

    def _tree_add(self, i, v):
        self._tadd(self.tree, i, v)

    def _prefix(self, i):
        return self._tpref(self.tree, i)

    # --- служебные ---
    def _w_of(self, s):
        return self.bweight[self.pos[s]] + self.extra.get(s, 0)

    def _excl_sorted(self, excl):
        """[(idx, weight)] отсортированно + префиксы."""
        items = sorted((self.pos[s], self._w_of(s)) for s in excl if s in self.pos)
        return items

    def _excl_below(self, items, idx):
        """Сумма исключённых весов с индексом позиции < idx."""
        # items отсортированы по idx; линейный проход (|excl| мал)
        s = 0
        for pi, w in items:
            if pi < idx:
                s += w
            else:
                break
        return s

    def _w(self, s):
        return self.bweight[self.pos[s]] + self.extra.get(s, 0)

    # --- контекстное взвешивание ---
    def _eff(self, ctx):
        """(тело, esc_rank, esc_lit) с множителем языка ctx."""
        if not self.split or ctx == 0:
            return self.raw_tot, self.esc_rank, self.esc_lit
        t = self.tree_cyr if ctx == 1 else self.tree_lat
        body = self.raw_tot + (self.split - 1) * self._tpref(t, len(self.bs))
        k = body // self.raw_tot
        return body, self.esc_rank * k, self.esc_lit * k

    def _we(self, idx, w, ctx):
        """Эффективный вес позиции idx в контексте ctx."""
        if ctx and self.split and self.lang[idx] == ctx:
            return w * self.split
        return w

    def _prefix_eff(self, i, ctx):
        p = self._tpref(self.tree, i)
        if not self.split or ctx == 0:
            return p
        t = self.tree_cyr if ctx == 1 else self.tree_lat
        return p + (self.split - 1) * self._tpref(t, i)

    def _excl_eff_sum(self, items, ctx):
        if not self.split or ctx == 0 or not items:
            return sum(w for _, w in items)
        s = 0
        for idx, w in items:
            s += w * self.split if self.lang[idx] == ctx else w
        return s

    def _excl_eff_below(self, items, idx, ctx):
        """Сумма эффективных весов исключённых с позицией < idx (idx 1-based)."""
        s = 0
        if not self.split or ctx == 0:
            for pi, w in items:
                if pi < idx:
                    s += w
                else:
                    break
            return s
        for pi, w in items:
            if pi < idx:
                s += w * self.split if self.lang[pi] == ctx else w
            else:
                break
        return s

    # --- API ---
    def total(self, ctx=0):
        body, escr, escl = self._eff(ctx)
        return body + escr + escl

    def total_excl(self, excl, ctx=0):
        body, escr, escl = self._eff(ctx)
        if not excl:
            return body + escr + escl
        s = self._excl_eff_sum(self._excl_sorted(excl), ctx)
        return body + escr + escl - s

    def in_table(self, sym, excl=None):
        if sym in self.bset:
            return not (excl and sym in excl)
        return False

    def reset(self):
        for idx, v in self.adds:
            self._tadd(self.tree, idx + 1, -v)
            if self.split:
                lg = self.lang[idx]
                if lg == 1:
                    self._tadd(self.tree_cyr, idx + 1, -v)
                elif lg == 2:
                    self._tadd(self.tree_lat, idx + 1, -v)
            self.raw_tot -= v
        self.adds.clear()
        self.extra.clear()
        self.esc_rank = self.esc_rank0
        self.esc_lit = self.esc_lit0

    def adapt(self, sym, insert=True, insert_sym=None):
        if sym == ESC_RANK:
            self.esc_rank = min(60000, self.esc_rank + ADAPT)
        elif sym == ESC_LIT:
            self.esc_lit = min(60000, self.esc_lit + ADAPT)
        elif sym in self.pos:
            w = self.extra.get(sym, 0)
            if w < 60000:
                idx = self.pos[sym]
                self._tadd(self.tree, idx + 1, ADAPT)
                if self.split:
                    lg = self.lang[idx]
                    if lg == 1:
                        self._tadd(self.tree_cyr, idx + 1, ADAPT)
                    elif lg == 2:
                        self._tadd(self.tree_lat, idx + 1, ADAPT)
                self.raw_tot += ADAPT
                self.adds.append((idx, ADAPT))
                self.extra[sym] = w + ADAPT
        return self

    def find_encode(self, sym, excl=None, ctx=0):
        """(cum, freq, tot); для ESC_RANK/ESC_LIT — хвостовые интервалы."""
        items = self._excl_sorted(excl) if excl else ()
        body, escr, escl = self._eff(ctx)
        base = body - self._excl_eff_sum(items, ctx)
        tot = base + escr + escl
        if sym == ESC_RANK:
            return base, escr, tot
        if sym == ESC_LIT:
            return base + escr, escl, tot
        if sym not in self.pos or (excl and sym in excl):
            # в кодеке не встречается (проверяется in_table/bset заранее);
            # деградируем в ESC-интервал, как линейный FT
            return base, escr, tot
        idx = self.pos[sym]
        w = self._w_of(sym)
        freq = self._we(idx, w, ctx)
        cum = self._prefix_eff(idx, ctx) - self._excl_eff_below(items, idx, ctx)
        return cum, freq, tot

    def decode_sym(self, cum, excl=None, ctx=0):
        """(sym|None, lo, freq); None = escape (ESC_RANK/ESC_LIT как None-
        ветка: возвращаем псевдосимвол)."""
        items = self._excl_sorted(excl) if excl else ()
        body, escr, escl = self._eff(ctx)
        base_tot = body - self._excl_eff_sum(items, ctx)
        if cum >= base_tot:
            if cum < base_tot + escr:
                return ESC_RANK, base_tot, escr
            return ESC_LIT, base_tot + escr, escl
        # бинарный поиск позиции: adj_prefix(p) > cum,
        # adj_prefix(p) = сумма весов позиций 0..p минус исключённые 0..p
        lo, hi = 0, len(self.bs) - 1
        while lo < hi:
            mid = (lo + hi) // 2
            ap = self._prefix_eff(mid + 1, ctx) - self._excl_eff_below(items, mid + 1, ctx)
            if ap > cum:
                hi = mid
            else:
                lo = mid + 1
        idx = lo
        s = self.bs[idx]
        w = self._w_of(s)
        freq = self._we(idx, w, ctx)
        lo_cum = self._prefix_eff(idx, ctx) - self._excl_eff_below(items, idx, ctx)
        return s, lo_cum, freq


class Model:
    def __init__(self, path=MODEL_PATH):
        with open(path, "rb") as f:
            data = f.read()
        assert data[:4] == b"SZGM" and data[4] == 2, "bad model (нужен SZGM)"
        r = Reader(data)
        r.p = 5
        self.D = r.varint()
        self.P = r.varint()
        self.L = r.varint()
        self.O0_TOP = r.varint()
        self.B1_ROWS = r.varint()
        self.B2_ROWS = r.varint()
        self.LEM_ROWS = r.varint()
        self.puncts = [r.blob().decode("utf-8") for _ in range(self.P)]
        self.words = [r.blob().decode("utf-8") for _ in range(self.D)]
        self.rank = {w: i for i, w in enumerate(self.words)}
        self.word_lang = [word_lang(w) for w in self.words]
        self.lemma_ids = [r.varint() for _ in range(self.D)]
        n0 = r.varint()
        syms, ws = [], []
        for _ in range(n0):
            syms.append(r.varint())
            ws.append(r.varint())
        esc_rank = r.varint()
        esc_lit = r.varint()
        o0_langs = [0 if s < self.P else self.word_lang[s - self.P] for s in syms]
        self.st_o0 = FenwickO0(syms, ws, esc_rank, esc_lit, o0_langs, O0_SPLIT)
        self.b1 = self._rows(r)
        self.b2 = self._rows(r)
        self.lem = self._rows(r)
        self.A = self._rows(r)
        self.C = self._rows(r)
        bw = [r.varint() for _ in range(256)]
        self.st_byte = FT(list(range(256)), bw, 0, astep=64, resize_at=10000)
        self.byte1_rows = {}
        for _ in range(r.varint()):
            pbi = r.varint()
            k = r.varint()
            syms, ws = [], []
            for _ in range(k):
                syms.append(r.varint())
                ws.append(r.varint())
            self.byte1_rows[pbi] = (syms, ws)
        self.st_case = []
        for _ in range(4):
            row = [r.varint() for _ in range(3)]
            self.st_case.append(FT([0, 1, 2], row, 0))
        lw = [r.varint() for _ in range(64)]
        self.st_len = FT(list(range(64)), lw, 0, astep=96)

    @staticmethod
    def _rows(r):
        d = {}
        for _ in range(r.varint()):
            k = r.varint()
            n = r.varint()
            syms, ws = [], []
            for _ in range(n):
                syms.append(r.varint())
                ws.append(r.varint())
            d[k] = FT(syms, ws, 40)
        return d
'''
TRAIN_SRC = r'''
# -*- coding: utf-8 -*-
"""
SZGC: сборка графовой модели сжатия SZGM.

Три потоковых прохода по корпусу (память ~1 ГБ, корпус до 800 МБ):
  P1: униграммы слов + byte0/byte1 статистики
      -> словарь D словоформ (blended: OpenSubtitles ru/en + корпус), леммы pymorphy3
  P2: биграммные пары (w2,w1) -> топ-B2_ROWS контекстов 2-го порядка
  P3: контексты (зеркало кодека): B1 (слово->слово), B2-продолжения,
      LEM (лемма->слово, согласование), A (слово->пунктуация), C (после punct),
      case/len статистики.

Модель SZGM: формат v2.
Выход: download/syllazip/graph-model.bin (+ stats json).
"""
import glob
import json
import os
import pickle
import re
import sys
import time
from collections import Counter, defaultdict

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from szgc_core import get_case, tokenize

BASE = os.environ.get("SZGC_BASE", os.path.dirname(os.path.abspath(__file__)))
CORP = os.path.join(BASE, "data/corpus")
OUT = os.path.join(BASE, "download/syllazip/graph-model.bin")
STATS = os.path.join(BASE, "download/syllazip/model-stats.json")
FREQ_RU = os.path.join(BASE, "scripts/ru_50k.txt")
FREQ_EN = os.path.join(BASE, "scripts/en_50k.txt")

def _env_int(name, default):
    try:
        return int(os.environ.get(name, default))
    except ValueError:
        return default


# ---- модельные ёмкости ----
PUNCT_TOP = 48
O0_TOP = _env_int("SZGC_O0_TOP", 131072)
DICT_CAP = _env_int("SZGC_DICT_CAP", 600000)
CTX_B1_ROWS = _env_int("SZGC_B1_ROWS", 65536)
B2_ROWS = _env_int("SZGC_B2_ROWS", 65536)
LEM_ROWS = _env_int("SZGC_LEM_ROWS", 16384)
A_ROWS = _env_int("SZGC_A_ROWS", 16384)
B1_KEEP, B2_KEEP, LEM_KEEP, A_KEEP, C_KEEP = 14, 9, 12, 4, 8
B1_MIN_CNT, B2_MIN_CNT, LEM_MIN_CNT, A_MIN_CNT = 40, 20, 30, 40
TOTAL_CAP = 3_000_000_000   # корпус до ~3 ГБ (RU+EN)
PAIR_SUBSAMPLE = 2          # брать каждую 2-ю пару (по счётчику)
ROW_TARGET = 1500           # суммарный вес статической строки контекста

# ---- английский контур: строки добавляются СВЕРХ русских квот ----
# (0 отключает соответствующую ручку; дефолты включены)
B2_EN_ADD = _env_int("SZGC_B2_EN_ADD", 32768)     # доп. en-en пары B2 сверх топа
B2_EN_MIN = _env_int("SZGC_B2_EN_MIN", 4)         # минимальный счёт доп. пары
B1_EN_ADD = _env_int("SZGC_B1_EN_ADD", 16384)     # доп. EN-слова в B1 сверх топа
A_EN_ADD = _env_int("SZGC_A_EN_ADD", 8192)        # доп. EN-слова в A сверх топа
LEM_EN_OFF = _env_int("SZGC_LEM_EN_OFF", 2)       # 2=убрать латинские леммы без дозаполнения, 1=с дозаполнением, 0=выкл
B1_KEEP_EN = _env_int("SZGC_B1_KEEP_EN", 32)      # глубина EN-строк B1
B2_KEEP_EN = _env_int("SZGC_B2_KEEP_EN", 16)      # глубина EN-строк B2
OUT = os.environ.get("SZGC_OUT", OUT)
STATS = os.environ.get("SZGC_STATS", STATS)

_LAT_W = re.compile(r"^[a-z]+$")


def is_latin(w):
    return _LAT_W.match(w) is not None


def read_freq_lists():
    ru, en = {}, {}
    for path, dst in ((FREQ_RU, ru), (FREQ_EN, en)):
        with open(path, encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= 50000:
                    break
                parts = line.strip().split(" ")
                if not parts or not parts[0]:
                    continue
                w = parts[0].strip().lower().replace("ё", "е").replace("Ё", "Е")
                n = int(parts[1].split()[0]) if len(parts) > 1 and parts[1].split()[0].isdigit() else 1
                if w:
                    dst[w] = dst.get(w, 0) + n
    return ru, en


def corpus_files():
    total, kept = 0, []
    for p in sorted(glob.glob(os.path.join(CORP, "*.txt"))):
        s = os.path.getsize(p)
        if total + s > TOTAL_CAP:
            break
        kept.append(p)
        total += s
    return kept, total


def iter_text(path, chunk=8 << 20):
    """Потоковое чтение файла кусками с безопасными границами (по \\n).
    Кусок без \\n накапливается. Статистический шум на ~75 стыках/файл."""
    with open(path, encoding="utf-8", errors="replace") as f:
        tail = ""
        while True:
            piece = f.read(chunk)
            if not piece:
                if tail:
                    yield tail
                return
            piece = tail + piece
            cut = piece.rfind("\n")
            if cut == -1:
                tail = piece
                continue
            tail = piece[cut:]
            yield piece[:cut]


def scale_weights(raw, target=ROW_TARGET):
    """Масштабирование весов: сумма ~target, предел 60000."""
    s = sum(raw.values())
    if s == 0 or target <= 0:
        return {}
    out = {}
    for k, v in raw.items():
        out[k] = max(1, min(60000, round(v * target / s)))
    budget = 2 * target + 64
    total = sum(out.values())
    if total > budget:
        # слишком много символов: сохраняем самых частых, редкие вылетают в ESC
        items = sorted(out.items(), key=lambda kv: (-kv[1], kv[0]))
        keep = {}
        acc = 0
        for k, w in items:
            if acc + w > budget and w == 1:
                break
            if acc + w > budget:
                out[k] = max(1, budget - acc)
                acc = budget
                break
            keep[k] = w
            acc += w
        if len(keep) < len(out):
            keep.update({k: out[k] for k in list(out)[len(keep):] if k in keep})
            out = keep
    return out


def varint(n):
    v = n & 0xFFFFFFFFFFFFFFFF
    bs = []
    while True:
        b = v & 0x7F
        v >>= 7
        if v:
            bs.append(b | 0x80)
        else:
            bs.append(b)
            return bytes(bs)


def build_puncts_from(counter_of_lits):
    ranked = [p for p, _ in counter_of_lits.most_common(PUNCT_TOP)]
    if " " not in ranked:
        ranked.insert(0, " ")
        ranked = ranked[:PUNCT_TOP]
    return ranked


# ============================================================== PASS 1
def pass1(files):
    t0 = time.time()
    cnt = Counter()
    byte0 = Counter()
    byte1 = [Counter() for _ in range(256)]
    lit_counter = Counter()
    prev_b = -1
    for fi, p in enumerate(files):
        for text in iter_text(p):
            for kind, tok in tokenize(text):
                if kind == "word":
                    cnt[tok.lower()] += 1
                    b = tok.encode("utf-8")
                    # байты слов участвуют в byte1-контексте (зеркалит поток вывода)
                    for x in b:
                        byte0[x] += 1
                        if prev_b >= 0:
                            byte1[prev_b][x] += 1
                        prev_b = x
                else:
                    if len(tok.encode("utf-8")) <= 16:
                        lit_counter[tok] += 1
                    lb = tok.encode("utf-8")
                    for x in lb:
                        byte0[x] += 1
                        if prev_b >= 0:
                            byte1[prev_b][x] += 1
                        prev_b = x
        if len(cnt) > 4_000_000:
            cnt = Counter({w: c for w, c in cnt.items() if c >= 2})
        print("  P1 %d/%d %s: слов-форм %d (%.0fs)" %
              (fi + 1, len(files), os.path.basename(p)[:24], len(cnt), time.time() - t0), flush=True)
    return cnt, byte0, byte1, lit_counter


def build_dictionary(cnt, ru_freq, en_freq):
    """Ранг словоформы -> sym. Blended-скор: частотники + корпус."""
    words = set(ru_freq) | set(en_freq)
    words.update(cnt.keys())
    latin = re.compile(r"^[a-z]+$")

    def score(w):
        s = cnt.get(w, 0) * 8 + (1 if cnt.get(w) else 0)
        s += ru_freq.get(w, 0)
        if latin.match(w):
            s += en_freq.get(w, 0)
        else:
            s += ru_freq.get(w.replace("ё", "е"), 0) if "ё" in w else 0
        return s

    ranked = sorted(words, key=lambda w: (-score(w), w))[:DICT_CAP]
    return ranked


def build_lemmas(ranked):
    """word -> lemma_idx; лемма-таблица только по словарю."""
    from pymorphy3 import MorphAnalyzer
    morph = MorphAnalyzer()
    latin = re.compile(r"^[a-z]+$")
    lemma_idx_of = {}
    lemma_ids = []
    lemmas = {}
    t0 = time.time()
    for i, w in enumerate(ranked):
        if w in lemma_idx_of:
            lemma_ids.append(lemma_idx_of[w])
            continue
        if latin.match(w):
            lem = w
        else:
            try:
                lem = morph.parse(w)[0].normal_form
            except Exception:
                lem = w
        if lem not in lemmas:
            lemmas[lem] = len(lemmas)
        lemma_idx_of[w] = lemmas[lem]
        lemma_ids.append(lemmas[lem])
        if i % 100000 == 0 and i:
            print("  леммы %d/%d (%.0fs)" % (i, len(ranked), time.time() - t0), flush=True)
    return lemma_ids, len(lemmas)


# ============================================================== PASS 2
def pass2(files, word_cnt, rank_of, P, lang_by_rank=None):
    """Пары (w2,w1) SYM-ов слов (как в кодеке); parity-субсэмпл."""
    t0 = time.time()
    buf = np.empty(1 << 23, dtype=np.int64)
    chunks = []
    bi = 0
    w1 = w2 = None
    gcnt = 0
    for p in files:
        for text in iter_text(p):
            for kind, tok in tokenize(text):
                if kind != "word":
                    continue
                low = tok.lower()
                r = rank_of.get(low, -1)
                if r >= 0 and w1 is not None and w2 is not None:
                    c1 = word_cnt.get(low, 0)
                    # фильтр частотности текущего слова (порог по униграммам)
                    if c1 >= B2_MIN_CNT:
                        if gcnt % PAIR_SUBSAMPLE == 0:
                            if bi >= buf.shape[0]:
                                chunks.append(buf[:bi])
                                buf = np.empty(1 << 23, dtype=np.int64)
                                bi = 0
                            buf[bi] = (w2 << 21) | w1
                            bi += 1
                if r >= 0:
                    w2, w1 = w1, P + r  # внесловарные слова не сбрасывают контекст
                gcnt += 1
        print("  P2 %s (%.0fs, пар %d)" % (os.path.basename(p)[:24], time.time() - t0, bi), flush=True)
    if bi:
        chunks.append(buf[:bi])
    del buf
    if not chunks:
        return set()
    allp = np.concatenate(chunks)
    del chunks
    uniq, counts = np.unique(allp, return_counts=True)
    del allp
    order = np.argsort(-counts, kind="stable")[:B2_ROWS]
    pairs = set(int(uniq[i]) for i in order)
    print("  P2: уникальных пар %d, отобрано %d (порог счёта >= %d)" %
          (len(uniq), len(pairs), int(counts[order[-1]]) if len(order) else 0), flush=True)
    # аддитивные en-en пары: сверх топа, русские строки не затрагиваются
    if B2_EN_ADD and lang_by_rank is not None:
        w1s = (uniq & ((1 << 21) - 1)).astype(np.int64)
        w2s = (uniq >> 21).astype(np.int64)
        en1 = lang_by_rank[w1s - P] == 2
        en2 = lang_by_rank[w2s - P] == 2
        cand = np.nonzero(en1 & en2 & (counts >= B2_EN_MIN))[0]
        top = set(int(x) for x in uniq[order])
        cand = np.array([i for i in cand if int(uniq[i]) not in top], dtype=np.int64)
        if len(cand) > B2_EN_ADD:
            # сортировка: счёт по убыванию, при равенстве — меньший ключ пары
            cand = cand[np.lexsort((uniq[cand], -counts[cand]))[:B2_EN_ADD]]
        added = set(int(uniq[i]) for i in cand)
        pairs |= added
        print("  P2: доп. en-en пар %d (счёт >= %d)" % (len(added), B2_EN_MIN), flush=True)
    return pairs


# ============================================================== PASS 3
def pass3(files, rank_of, puncts, pairs, b1_ctx, lem_ctx, lemma_ids, P):
    """Все контекстные счётчики (зеркало состояния кодека)."""
    t0 = time.time()
    b2 = defaultdict(Counter)
    b1 = defaultdict(Counter)
    lem = defaultdict(Counter)
    A = defaultdict(Counter)
    C = defaultdict(Counter)
    case_cnt = [[0, 0, 0] for _ in range(4)]
    len_cnt = Counter()
    lit_cnt = Counter()
    punct_set = set(puncts)
    punct_idx = {p: i for i, p in enumerate(puncts)}
    prev = None                  # ("word", sym) | ("punct", idx) | ("lit",)
    w1 = w2 = None               # последние два sym слов (через файлы и куски)

    for p in files:
        for text in iter_text(p):
            for kind, tok in tokenize(text):
                if kind == "word":
                    low = tok.lower()
                    r = rank_of.get(low)
                    flag = get_case(tok)
                    if r is not None and flag != 3:
                        sym = P + r
                        # --- прогноз этого слова доступными контекстами ---
                        if w1 is not None and w2 is not None:
                            pr = (w2 << 21) | w1
                            if pr in pairs:
                                b2[pr][sym] += 1
                        if w1 is not None:
                            li = lemma_ids[w1 - P]
                            if li in lem_ctx:
                                lem[li][sym] += 1
                            if prev is not None and prev[0] == "punct":
                                if puncts[prev[1]] == " ":
                                    if w1 in b1_ctx:
                                        b1[w1][sym] += 1
                                else:
                                    C[prev[1]][sym] += 1
                        # --- регистр ---
                        cctx = 0
                        if prev is None:
                            cctx = 0
                        elif prev[0] == "punct":
                            s = puncts[prev[1]]
                            cctx = 1 if (s and s.rstrip()[-1:] in (".", "!", "?", "…")) else 2
                        else:
                            cctx = 3
                        case_cnt[cctx][flag] += 1
                        w2, w1 = w1, sym
                        prev = ("word", sym)
                    else:
                        # слово вне словаря -> литеральный прогон
                        lit_cnt[tok.encode("utf-8")] += 1
                        lb = tok.encode("utf-8")
                        i = 0
                        while i < len(lb):
                            k = len(lb) - i
                            ln = 63 if k > 63 else k
                            len_cnt[min(ln, 63)] += 1
                            i += ln
                        prev = ("lit",)
                        # w1/w2 НЕ сбрасываем (зеркало кодека)
                else:
                    if tok in punct_set:
                        pidx = punct_idx[tok]
                        if prev is not None and prev[0] == "word":
                            # A: пунктуация после слова
                            if w1 is not None:
                                A[w1][pidx] += 1
                        elif prev is not None and prev[0] == "punct":
                            C[prev[1]][pidx] += 1
                        prev = ("punct", pidx)
                    else:
                        lit_cnt[tok.encode("utf-8")] += 1
                        lb = tok.encode("utf-8")
                        i = 0
                        while i < len(lb):
                            k = len(lb) - i
                            ln = 63 if k > 63 else k
                            len_cnt[min(ln, 63)] += 1
                            i += ln
                        prev = ("lit",)
        print("  P3 %s (%.0fs)" % (os.path.basename(p)[:24], time.time() - t0), flush=True)

    return b2, b1, lem, A, C, case_cnt, len_cnt, lit_cnt


STAGE_DIR = os.path.join(BASE, "data/stage2")


def stage1():
    """P1: униграммы + байты + лит-токены."""
    os.makedirs(STAGE_DIR, exist_ok=True)
    t00 = time.time()
    files, total_bytes = corpus_files()
    print("файлов: %d, корпус: %.1f МБ" % (len(files), total_bytes / 1048576))
    cnt, byte0, byte1, lit_counter = pass1(files)
    with open(os.path.join(STAGE_DIR, "stage1.pkl"), "wb") as f:
        pickle.dump({"cnt": cnt, "byte0": byte0, "byte1": byte1,
                     "lit_counter": lit_counter, "total_bytes": total_bytes},
                    f, protocol=4)
    print("stage1 OK (%.0fs): словоформ %d" % (time.time() - t00, len(cnt)))


def stage2():
    """Словарь + леммы + puncts + контекстные множества."""
    os.makedirs(STAGE_DIR, exist_ok=True)
    t00 = time.time()
    with open(os.path.join(STAGE_DIR, "stage1.pkl"), "rb") as f:
        s1 = pickle.load(f)
    cnt, byte0, byte1, lit_counter = s1["cnt"], s1["byte0"], s1["byte1"], s1["lit_counter"]
    ru_freq, en_freq = read_freq_lists()
    ranked = build_dictionary(cnt, ru_freq, en_freq)
    D = len(ranked)
    rank_of = {w: i for i, w in enumerate(ranked)}
    puncts = build_puncts_from(lit_counter)
    P = len(puncts)
    print("словарь D=%d, puncts P=%d" % (D, P))
    lemma_ids, L = build_lemmas(ranked)
    print("лемм L=%d (%.0fs)" % (L, time.time() - t00))
    lemma_agg = Counter()
    latin_lem = set()
    for i, li in enumerate(lemma_ids):
        lemma_agg[li] += cnt.get(ranked[i], 0)
        if is_latin(ranked[i]):
            latin_lem.add(li)
    if LEM_EN_OFF == 2:
        # убрать латинские леммы без дозаполнения: русский набор строк
        # в точности как в смешанном топе, глубокие леммы не подмешиваются
        cand = [li for li, c in lemma_agg.items() if c >= LEM_MIN_CNT]
        sel = sorted(cand, key=lambda x: (-lemma_agg[x], x))[:LEM_ROWS]
        lem_ctx = set(li for li in sel if li not in latin_lem)
    else:
        if LEM_EN_OFF:
            for li in list(lemma_agg):
                if li in latin_lem:
                    del lemma_agg[li]
        lem_ctx = set(li for li, c in lemma_agg.items() if c >= LEM_MIN_CNT)
        lem_ctx = set(sorted(lem_ctx, key=lambda x: -lemma_agg[x])[:LEM_ROWS])
    freq_words = [(w, c) for w, c in cnt.items() if c >= B1_MIN_CNT]
    freq_words.sort(key=lambda t: (-t[1], t[0]))
    b1_ctx = set(P + rank_of[w] for w, _ in freq_words[:CTX_B1_ROWS] if w in rank_of)
    a_ctx = set(P + rank_of[w] for w, _ in freq_words[:A_ROWS] if w in rank_of)
    # аддитивные EN-контексты: сверх топа, русские строки не затрагиваются
    if B1_EN_ADD:
        add = [w for w, _ in freq_words[CTX_B1_ROWS:] if is_latin(w)][:B1_EN_ADD]
        b1_ctx |= set(P + rank_of[w] for w in add if w in rank_of)
    if A_EN_ADD:
        add = [w for w, _ in freq_words[A_ROWS:] if is_latin(w)][:A_EN_ADD]
        a_ctx |= set(P + rank_of[w] for w in add if w in rank_of)
    print("контекстов: b1=%d lem=%d a=%d" % (len(b1_ctx), len(lem_ctx), len(a_ctx)))
    with open(os.path.join(STAGE_DIR, "stage2.pkl"), "wb") as f:
        pickle.dump({"ranked": ranked, "puncts": puncts, "lemma_ids": lemma_ids,
                     "L": L, "lem_ctx": lem_ctx, "b1_ctx": b1_ctx,
                     "a_ctx": a_ctx, "byte0": byte0, "byte1": byte1}, f, protocol=4)
    print("stage2 OK (%.0fs)" % (time.time() - t00))


def stage3():
    """P2: отбор биграммных контекстов."""
    t00 = time.time()
    with open(os.path.join(STAGE_DIR, "stage2.pkl"), "rb") as f:
        s2 = pickle.load(f)
    ranked = s2["ranked"]
    rank_of = {w: i for i, w in enumerate(ranked)}
    with open(os.path.join(STAGE_DIR, "stage1.pkl"), "rb") as f:
        cnt = pickle.load(f)["cnt"]
    files, _ = corpus_files()
    with open(os.path.join(STAGE_DIR, "stage2.pkl"), "rb") as f:
        P = len(pickle.load(f)["puncts"])
    lang_by_rank = np.array([2 if is_latin(w) else 0 for w in ranked], dtype=np.int8)
    pairs = pass2(files, cnt, rank_of, P, lang_by_rank)
    import numpy as _np
    arr = _np.array(sorted(pairs), dtype=_np.int64)
    _np.save(os.path.join(STAGE_DIR, "pairs.npy"), arr)
    print("stage3 OK (%.0fs): пар %d" % (time.time() - t00, len(pairs)))


def stage4():
    """P3 + упаковка модели SZGM."""
    t00 = time.time()
    with open(os.path.join(STAGE_DIR, "stage2.pkl"), "rb") as f:
        s2 = pickle.load(f)
    ranked, puncts = s2["ranked"], s2["puncts"]
    lemma_ids, L = s2["lemma_ids"], s2["L"]
    lem_ctx, b1_ctx, a_ctx = s2["lem_ctx"], s2["b1_ctx"], s2["a_ctx"]
    byte0, byte1 = s2["byte0"], s2["byte1"]
    rank_of = {w: i for i, w in enumerate(ranked)}
    D = len(ranked)
    P = len(puncts)
    import numpy as _np
    pairs = set(int(x) for x in _np.load(os.path.join(STAGE_DIR, "pairs.npy")))
    files, total_bytes = corpus_files()
    print("== P3: контексты ==")
    b2, b1, lem, A, C, case_cnt, len_cnt, lit_cnt = pass3(
        files, rank_of, puncts, pairs, b1_ctx, lem_ctx, lemma_ids, P)
    with open(os.path.join(STAGE_DIR, "stage1.pkl"), "rb") as f:
        cnt = pickle.load(f)["cnt"]
    ru_freq, en_freq = read_freq_lists()
    pack_model(cnt, ru_freq, ranked, puncts, lemma_ids, L, lem_ctx, b1_ctx, a_ctx,
               pairs, b2, b1, lem, A, C, case_cnt, len_cnt, byte0, byte1,
               total_bytes, len(files), t00, en_freq=en_freq)


def pack_model(cnt, ru_freq, ranked, puncts, lemma_ids, L, lem_ctx, b1_ctx, a_ctx,
               pairs, b2, b1, lem, A, C, case_cnt, len_cnt, byte0, byte1,
               total_bytes, nfiles, t00, en_freq=None):
    D = len(ranked)
    P = len(puncts)
    rank_of = {w: i for i, w in enumerate(ranked)}
    latin_rank = set(i for i, w in enumerate(ranked) if is_latin(w))
    print("== упаковка модели ==")
    out = bytearray()
    out += b"SZGM"
    out.append(2)
    wv = lambda n: out.extend(varint(n))
    wv(D); wv(P); wv(L); wv(O0_TOP); wv(CTX_B1_ROWS); wv(B2_ROWS); wv(LEM_ROWS)
    for p in puncts:
        b = p.encode("utf-8")
        wv(len(b)); out += b
    for w in ranked:
        b = w.encode("utf-8")
        wv(len(b)); out += b
    for li in lemma_ids:
        wv(li)

    # order-0
    o0_raw = Counter()
    for w, c in cnt.items():
        r = rank_of.get(w)
        if r is not None:
            o0_raw[P + r] += c
    for w, c in ru_freq.items():
        r = rank_of.get(w) or rank_of.get(w.replace("ё", "е"))
        if r is not None:
            o0_raw[P + r] += 1
    if en_freq:
        for w, c in en_freq.items():
            r = rank_of.get(w)
            if r is not None:
                o0_raw[P + r] += max(1, min(60, c // 500))
    o0_syms = sorted(s for s in o0_raw if s < O0_TOP)
    o0_syms = sorted(set(o0_syms) | set(range(P)))[:O0_TOP]
    o0_w = scale_weights({s: max(o0_raw.get(s, 0), 1) for s in o0_syms},
                        max(16000, len(o0_syms)))
    wv(len(o0_w))
    for s in sorted(o0_w):
        wv(s); wv(o0_w[s])
    wv(120); wv(60)      # esc_rank, esc_lit (адаптируются кодеком)

    # B1
    rows = []
    for ctx in sorted(b1_ctx):
        c = b1.get(ctx)
        if c and len(c) >= 2:
            keep = B1_KEEP_EN if (ctx - P) in latin_rank else B1_KEEP
            rows.append((ctx, scale_weights(dict(c.most_common(keep)))))
    wv(len(rows))
    for ctx, w in rows:
        wv(ctx); wv(len(w))
        for s in sorted(w):
            wv(s); wv(w[s])
    print("B1 строк: %d" % len(rows))

    # B2 (однозначные продолжения — самый ценный случай, не выбрасывать)
    rows = []
    for pr in sorted(pairs):
        c = b2.get(pr)
        if c and (len(c) >= 2 or sum(c.values()) >= 3):
            keep = B2_KEEP_EN if ((pr & ((1 << 21) - 1)) - P) in latin_rank else B2_KEEP
            rows.append((pr, scale_weights(dict(c.most_common(keep)))))
    wv(len(rows))
    for pr, w in rows:
        wv(pr); wv(len(w))
        for s in sorted(w):
            wv(s); wv(w[s])
    print("B2 строк: %d" % len(rows))

    # LEM
    rows = []
    for li in sorted(lem_ctx):
        c = lem.get(li)
        if c and len(c) >= 2:
            rows.append((li, scale_weights(dict(c.most_common(LEM_KEEP)))))
    wv(len(rows))
    for li, w in rows:
        wv(li); wv(len(w))
        for s in sorted(w):
            wv(s); wv(w[s])
    print("LEM строк: %d" % len(rows))

    # A
    rows = []
    for ctx in sorted(a_ctx):
        c = A.get(ctx)
        if c:
            rows.append((ctx, scale_weights(dict(c.most_common(A_KEEP)))))
    wv(len(rows))
    for ctx, w in rows:
        wv(ctx); wv(len(w))
        for s in sorted(w):
            wv(s); wv(w[s])
    print("A строк: %d" % len(rows))

    # C
    rows = []
    for pidx in range(P):
        c = C.get(pidx)
        if c:
            rows.append((pidx, scale_weights(dict(c.most_common(C_KEEP)))))
    wv(len(rows))
    for pidx, w in rows:
        wv(pidx); wv(len(w))
        for s in sorted(w):
            wv(s); wv(w[s])

    # byte0 / byte1
    bw = [max(1, min(600, byte0.get(b, 0))) for b in range(256)]
    s = sum(bw)
    bw = [max(1, w * 6000 // s) for w in bw]
    for b in bw:
        wv(b)
    n_b1 = 0
    rows = []
    for pbi in range(256):
        c = byte1[pbi]
        if c:
            rows.append((pbi, scale_weights(dict(c.most_common(16)), 900)))
    wv(len(rows))
    for pbi, w in rows:
        wv(pbi); wv(len(w))
        for x in sorted(w):
            wv(x); wv(w[x])

    # case / len
    for row in case_cnt:
        ssum = max(1, sum(row))
        for x in row:
            wv(max(1, min(60000, x * 1000 // ssum)))
    lw = [max(1, 4000 >> min(i, 20)) for i in range(64)]
    lw[0] = 900
    ssum = sum(len_cnt.values())
    if ssum:
        lw = [max(1, min(60000, len_cnt.get(i, 0) * 4000 // ssum)) for i in range(64)]
    for x in lw:
        wv(x)

    os.makedirs(os.path.dirname(OUT), exist_ok=True)
    with open(OUT, "wb") as f:
        f.write(bytes(out))
    print("модель: %s (%.2f МБ, всего %.0fs)" % (OUT, len(out) / 1048576, time.time() - t00))

    stats = {
        "corpus_bytes": total_bytes,
        "files": nfiles,
        "dict_D": D,
        "puncts": P,
        "lemmas": L,
        "o0": len(o0_w),
        "b1_rows": len(b1_ctx),
        "b2_pairs": len(pairs),
        "lem_rows": len(lem_ctx),
        "model_bytes": len(out),
        "top_words": ranked[:30],
    }
    with open(STATS, "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=1)
    print("stats ->", STATS)


if __name__ == "__main__":
    arg = sys.argv[1] if len(sys.argv) > 1 else "all"
    if arg == "stage1":
        stage1()
    elif arg == "stage2":
        stage2()
    elif arg == "stage3":
        stage3()
    elif arg == "stage4":
        stage4()
    else:
        t0 = time.time()
        stage1()
        stage2()
        stage3()
        stage4()
        print("ВСЕГО: %.0fs" % (time.time() - t0))
'''


In [ ]:
# ---------- 2. Код тренера ----------
# szgc_core пишется первым: build_model импортирует из него tokenize/get_case
with open('/content/szgc/szgc_core.py', 'w') as f:
    f.write(CORE_SRC)
with open('/content/szgc/build_model.py', 'w') as f:
    f.write(TRAIN_SRC)
CORE_SRC = TRAIN_SRC = None  # освобождаем

# пути Colab: перенаправляем корпус и результаты из репозитория в /content
p = '/content/szgc/build_model.py'
src = open(p).read()
for old, new in [
    ('BASE = "/home/z/my-project"', 'BASE = "/content"'),
    ('BASE = os.environ.get("SZGC_BASE", os.path.dirname(os.path.abspath(__file__)))', 'BASE = "/content"'),
    ('CORP = os.path.join(BASE, "data/corpus")', 'CORP = "/content/data/corpus"'),
    ('OUT = os.path.join(BASE, "download/syllazip/graph-model.bin")', 'OUT = "/content/graph-model.bin"'),
    ('STATS = os.path.join(BASE, "download/syllazip/model-stats.json")', 'STATS = "/content/model-stats.json"'),
    ('FREQ_RU = os.path.join(BASE, "scripts/ru_50k.txt")', 'FREQ_RU = "/content/ru_50k.txt"'),
    ('FREQ_EN = os.path.join(BASE, "scripts/en_50k.txt")', 'FREQ_EN = "/content/en_50k.txt"'),
    ('STAGE_DIR = "/home/z/my-project/data/stage2"', 'STAGE_DIR = "/content/stage"'),
    ('STAGE_DIR = os.path.join(BASE, "data/stage2")', 'STAGE_DIR = "/content/stage"'),
]:
    src = src.replace(old, new)
assert 'BASE = "/content"' in src and 'OUT = "/content/graph-model.bin"' in src
open(p, 'w').write(src)

# ---------- 3. Обучение (4 стадии, можно запускать по отдельности) ----------
import sys, time
sys.path.insert(0, '/content/szgc')
import build_model as B
t0 = time.time()
B.stage1(); B.stage2(); B.stage3(); B.stage4()
print('Обучение завершено: %.0f мин' % ((time.time() - t0) / 60))


In [ ]:
CODEC_SRC = r'''
# -*- coding: utf-8 -*-
"""
SZGC — морфографовый предиктивный кодек (SyllaZip Graph Coder).

Поток: [kind-флаг: слово|punct|литерал] для каждого токена.
  слово : MATCH -> O2 (биграмма w2,w1) -> B1 (w1->слово, gap " ") -> C (после punct)
          -> LEM (лемма w1: согласование) -> O0 -> ESC_RANK (varint-ранг)
  punct : A (слово->punct) / C (после punct) -> O0
  литерал: escape контекста -> чанки len-модели -> байты (order-1 по потоку вывода)

PPMC-семантика: escape из таблицы -> её символы исключаются (exclusions) из
младших порядков; вес escape и вставка символа применяются ПОСЛЕ того, как
символ стал известен
"""
import glob
import os
import random
import sys
import time

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from szgc_core import (ADAPT, B_ASTEP, B_ESC, CTX_CAP, ESC_LIT,
                        ESC_RANK, FT, M32, MODEL_PATH, Model, PAIR_SHIFT,
                        RangeDecoder, RangeEncoder, TOP, W1_O0, W1_ROW,
                        W1_FORCE, W_MATCH, ESC_MATCH, W_LZP_HIT, W_LZP_MISS,
                        apply_case, get_case, put_varint, tokenize)

_FMT_TAG = 2        # формат-тег потока (первый байт)


class SZGC2:
    def __init__(self, model):
        self.m = model
        if not hasattr(model, "touched"):
            model.touched = set()     # общее для всех инстансов: мутирует общая модель
        self.match = {}
        self.b1c = {}                 # order-1 строки: prev_byte -> FT (esc=0)
        self.b1hit = {}               # order-1 ternary-флаги: prev_byte -> FT
        self.bctx = {}                # order -> {ключ-байты: FT} (чисто адаптивные)
        self.bhist = []               # история байт выходного потока (контексты order-1..3)
        self.byte_mode = None         # None: токенный; 0/1/2: байтовый (глубина)
        self._bo0 = None              # масштабированная o0 для байтовых режимов
        self.kindc = {}               # kind-таблицы по предыдущему kind (биграммы)
        self.last_kind = 0
        self.prev = None              # ("word", sym) | ("punct", idx) | ("lit",)
        self.prev2 = None
        self.w1 = None                # последний sym слова
        self.w2 = None                # предпоследний sym слова
        self.wlist = []               # история sym-ов слов (для LZP)
        self.lzp_pos = {}             # pair -> индекс прошлого продолжения в wlist
        self.lzp_flag = FT([1, 0], [W_LZP_HIT, W_LZP_MISS], 0)
        self.raw_t = FT([0, 1, 2], [64, 96, 40], 0)
        # 0=сырой 1 байт 1=структурный 2=сырой прогон

    # ------------------------------------------------------------ state
    def _fresh(self):
        for ft in self.m.touched:
            ft.reset()
        self.m.touched.clear()
        self.match.clear()
        self.b1c.clear()
        self.b1hit.clear()
        self.bctx.clear()
        self.bhist = []
        if self._bo0 is not None:
            self._bo0.reset()
        self.prev = self.prev2 = None
        self.w1 = self.w2 = None
        self.wlist = []
        self.lzp_pos = {}
        self.lzp_flag.reset()
        self.kindc.clear()
        self.last_kind = 0
        self.raw_t.reset()

    @staticmethod
    def _t(ft):
        return ft

    def _byte1_row(self, pb):
        """Статическая строка order-1 без escape (esc=0); hit решается отдельным битом."""
        ft = self.b1c.get(pb)
        if ft is None:
            row = self.m.byte1_rows.get(pb)
            if row is not None:
                ft = FT(row[0], row[1], 0, astep=B_ASTEP)
            else:
                ft = FT(None, None, 0, astep=B_ASTEP)
            self.b1c[pb] = ft
        return ft

    def _byte1_hit_ft(self, pb):
        """Адаптивный ternary-флаг order-1: 0=мимо строки в o0, 1=строка, 2=строка не знает."""
        ft = self.b1hit.get(pb)
        if ft is None:
            ft = FT([0, 1, 2], [W1_O0, W1_ROW, W1_FORCE], 0)
            self.b1hit[pb] = ft
        return ft

    def _byte_ctx_ft(self, order, key):
        """Чисто адаптивная строка контекста order-2/3 (создаётся на лету)."""
        d = self.bctx.setdefault(order, {})
        ft = d.get(key)
        if ft is None:
            ft = FT(None, None, B_ESC, astep=B_ASTEP, cap=256)
            d[key] = ft
        return ft

    def _byte_tables(self, hist=None):
        """Цепочка чисто адаптивных байтовых контекстов; глубина по режиму:
        None/0 — order-3 -> order-2; 1 — order-2; 2 — без цепочки.
        order-1 (только токенный режим) обрабатывается отдельно, терминал — st_byte.
        hist=None — рабочая история; иначе оценка на копии (без мутаций)."""
        h = self.bhist if hist is None else hist
        bm = self.byte_mode
        tabs = []
        if bm != 2:
            if (bm is None or bm == 0) and len(h) >= 3:
                tabs.append(self._byte_ctx_ft(3, (h[-3], h[-2], h[-1])))
            if len(h) >= 2:
                tabs.append(self._byte_ctx_ft(2, (h[-2], h[-1])))
        return tabs

    def _byte_o0_ft(self):
        """o0 для байтовых режимов: base>>4 (слабый приор ~6 псевдосчётов),
        быстрая адаптация — алфавит мусора выучивается за цикл."""
        ft = self._bo0
        if ft is None:
            bw = [max(1, w >> 4) for w in self.m.st_byte.bw]
            ft = FT(list(range(256)), bw, 0, astep=64, resize_at=3000)
            self._bo0 = ft
        return ft

    def _term_o0(self):
        """Терминальная o0: статическая в токенном режиме, масштабированная в байтовых."""
        return self.m.st_byte if self.byte_mode is None else self._byte_o0_ft()

    def _enc_byte(self, enc, b):
        excl = set()
        escaped = []
        b1_miss = [None]
        done = False
        for ft in self._byte_tables():
            if ft.in_table(b, excl):
                cum, freq, tot = ft.find_encode(b, excl)
                enc.encode(cum, freq, tot)
                ft.adapt(b, insert=False)
                done = True
                break
            cum, freq, tot = ft.find_encode(None, excl)
            enc.encode(cum, freq, tot)
            escaped.append(ft)
            excl.update(ft.bs)
            excl.update(ft.delta.keys())
        if not done and self.bhist and self.byte_mode is None:
            # order-1: ternary-флаг + строка без escape; выбор строка/o0
            hit_t = self._byte1_hit_ft(self.bhist[-1])
            row = self._byte1_row(self.bhist[-1])
            inrow = row.in_table(b, excl) and row.total_excl(excl) > 0
            if inrow:
                # сравнение: строка (флаг 1 + sym) против o0 (флаг 0 + sym)
                c1, f1, t1 = hit_t.find_encode(1)
                cr, fr, tr = row.find_encode(b, excl)
                c0, f0, t0 = hit_t.find_encode(0)
                co, fo, to = self._term_o0().find_encode(b, excl)
                cost_row = ((t1.bit_length() - f1.bit_length()) +
                            (tr.bit_length() - fr.bit_length()))
                cost_o0 = ((t0.bit_length() - f0.bit_length()) +
                           (to.bit_length() - fo.bit_length()))
                v = 1 if cost_row <= cost_o0 else 0
            else:
                v = 2
            cum, freq, tot = hit_t.find_encode(v)
            enc.encode(cum, freq, tot)
            hit_t.adapt(v, insert=False)
            if v == 1:
                cum, freq, tot = row.find_encode(b, excl)
                enc.encode(cum, freq, tot)
                row.adapt(b, insert=False)
                done = True
            else:
                if v == 2:
                    b1_miss[0] = row
        if not done:
            t = self._term_o0()
            if not t.total_excl(excl):
                excl = set()          # все 256 исключены — терминал без исключений
            cum, freq, tot = t.find_encode(b, excl)
            enc.encode(cum, freq, tot)
            t.adapt(b, insert=False)
            self.m.touched.add(t)
        for ft in escaped:
            ft.adapt(None, insert=True, insert_sym=b)
        if b1_miss[0] is not None:
            b1_miss[0].adapt(None, insert=True, insert_sym=b)
        self.bhist.append(b)

    def _dec_byte(self, dec):
        excl = set()
        escaped = []
        b1_miss = [None]
        sym = None
        for ft in self._byte_tables():
            cum = dec.start(ft.total_excl(excl))
            s, lo, freq = ft.decode_sym(cum, excl)
            dec.finish(lo, freq)
            if s is not None:
                ft.adapt(s, insert=False)
                sym = s
                break
            escaped.append(ft)
            excl.update(ft.bs)
            excl.update(ft.delta.keys())
        if sym is None and self.bhist and self.byte_mode is None:
            hit_t = self._byte1_hit_ft(self.bhist[-1])
            row = self._byte1_row(self.bhist[-1])
            cum = dec.start(hit_t.total())
            v, lo, freq = hit_t.decode_sym(cum)
            dec.finish(lo, freq)
            hit_t.adapt(v, insert=False)
            if v == 1:
                cum = dec.start(row.total_excl(excl))
                s2, lo, freq = row.decode_sym(cum, excl)
                dec.finish(lo, freq)
                row.adapt(s2, insert=False)
                sym = s2
            elif v == 2:
                b1_miss[0] = row
        if sym is None:
            t = self._term_o0()
            if not t.total_excl(excl):
                excl = set()
            cum = dec.start(t.total_excl(excl))
            s, lo, freq = t.decode_sym(cum, excl)
            dec.finish(lo, freq)
            t.adapt(s, insert=False)
            self.m.touched.add(t)
            sym = s
        for ft in escaped:
            ft.adapt(None, insert=True, insert_sym=sym)
        if b1_miss[0] is not None:
            b1_miss[0].adapt(None, insert=True, insert_sym=sym)
        self.bhist.append(sym)
        return sym

    # ------------------------------------------------- контекст punct/lit
    def _ctx_ac(self):
        """Таблица A/C для punct-прогноза и escape литералов (v1-семантика)."""
        prev = self.prev
        if prev is None:
            return None
        if prev[0] == "word":
            return self.m.A.get(prev[1])
        if prev[0] == "punct":
            return self.m.C.get(prev[1])
        return None

    def _case_ctx(self):
        prev = self.prev
        if prev is None:
            return 0
        if prev[0] == "punct":
            s = self.m.puncts[prev[1]]
            return 1 if (s and s.rstrip()[-1:] in (".", "!", "?", "…")) else 2
        return 3

    def _enc_varint(self, enc, n):
        buf = bytearray()
        put_varint(buf, n)
        for b in buf:
            self._enc_byte(enc, b)

    def _dec_varint(self, dec):
        v, sh = 0, 0
        while True:
            b = self._dec_byte(dec)
            v |= (b & 0x7F) << sh
            sh += 7
            if not (b & 0x80):
                return v

    def _enc_lit(self, enc, data):
        i = 0
        while i < len(data):
            k = len(data) - i
            if k > 63:
                chunk, s = data[i:i + 63], 62
            elif k == 63:
                chunk, s = data[i:i + 63], 63
            else:
                chunk, s = data[i:i + k], k - 1
            t = self.m.st_len
            cum, freq, tot = t.find_encode(s)
            enc.encode(cum, freq, tot)
            t.adapt(s, insert=False)
            self.m.touched.add(t)
            for b in chunk:
                self._enc_byte(enc, b)
            i += len(chunk)

    def _dec_lit(self, dec):
        out = bytearray()
        while True:
            t = self.m.st_len
            cum = dec.start(t.total())
            s, lo, freq = t.decode_sym(cum)
            dec.finish(lo, freq)
            t.adapt(s, insert=False)
            self.m.touched.add(t)
            n = 63 if s >= 62 else s + 1
            for _ in range(n):
                out.append(self._dec_byte(dec))
            if s != 62:
                return bytes(out)

    def _enc_case(self, enc, flag, ctx):
        t = self.m.st_case[ctx]
        cum, freq, tot = t.find_encode(flag)
        enc.encode(cum, freq, tot)
        t.adapt(flag, insert=False)
        self.m.touched.add(t)

    def _dec_case(self, dec, ctx):
        t = self.m.st_case[ctx]
        cum = dec.start(t.total())
        s, lo, freq = t.decode_sym(cum)
        dec.finish(lo, freq)
        t.adapt(s, insert=False)
        self.m.touched.add(t)
        return s

    # --------------------------------------------- оценки стоимости (выбор пути)
    # Стоимость шага кодера в 1/256 бита: bit_length(tot)-bit_length(freq).

    def _est_bits(self, acc, cum, freq, tot):
        if freq > 0:
            acc[0] += (tot.bit_length() - freq.bit_length()) * 256

    def _est_ft(self, acc, ft, sym, excl, ctx=0):
        """Оценка кодирования sym в ft; False = escape (excl пополняется)."""
        if ft is None:
            return False
        if ft.in_table(sym, excl):
            cum, freq, tot = ft.find_encode(sym, excl, ctx)
            self._est_bits(acc, cum, freq, tot)
            return True
        cum, freq, tot = ft.find_encode(None, excl, ctx)
        self._est_bits(acc, cum, freq, tot)
        if excl is not None:
            excl.update(ft.bs)
            excl.update(ft.delta.keys())
        return False

    def _est_byte(self, b, hist):
        """Оценка стоимости байта: o3 -> o2 -> o1(hit-бит) -> st_byte.
        hist — локальная копия истории"""
        acc = [0]
        excl = set()
        hit = False
        for ft in self._byte_tables(hist):
            if self._est_ft(acc, ft, b, excl):
                hit = True
                break
        if not hit and hist:
            hit_t = self._byte1_hit_ft(hist[-1])
            row = self._byte1_row(hist[-1])
            inrow = row.in_table(b, excl) and row.total_excl(excl) > 0
            t = self._term_o0()
            if not t.total_excl(excl):
                excl = set()
            co, fo, to = t.find_encode(b, excl)
            if inrow:
                cr, fr, tr = row.find_encode(b, excl)
                c1, f1, t1 = hit_t.find_encode(1)
                c0, f0, t0 = hit_t.find_encode(0)
                cost_row = ((t1.bit_length() - f1.bit_length()) +
                            (tr.bit_length() - fr.bit_length()))
                cost_o0 = ((t0.bit_length() - f0.bit_length()) +
                           (to.bit_length() - fo.bit_length()))
                v = 1 if cost_row <= cost_o0 else 0
            else:
                v = 2
            cv, fv, tv = hit_t.find_encode(v)
            self._est_bits(acc, cv, fv, tv)
            if v == 1:
                self._est_bits(acc, cr, fr, tr)
            else:
                self._est_bits(acc, co, fo, to)
        elif not hit:
            t = self._term_o0()
            if not t.total_excl(excl):
                excl = set()
            cum, freq, tot = t.find_encode(b, excl)
            self._est_bits(acc, cum, freq, tot)
        return acc[0]

    def _est_varint(self, n, hist):
        buf = bytearray()
        put_varint(buf, n)
        acc = 0
        for b in buf:
            acc += self._est_byte(b, hist)
            hist.append(b)
        return acc

    def _est_chain(self, sym, flag, pred, r):
        """Оценка словарной цепочки для слова (escape-каскад + case)."""
        acc = [0]
        excl = set()
        ctx = self._o0_ctx()
        if pred is not None:
            t = self.lzp_flag
            cum, freq, tot = t.find_encode(0)
            self._est_bits(acc, cum, freq, tot)
            excl.add(pred)
        hit = False
        for ft, tag in self._word_tables():
            if self._est_ft(acc, ft, sym, excl, ctx):
                hit = True
                break
        if not hit:
            o0 = self.m.st_o0
            if not self._est_ft(acc, o0, sym, excl, ctx):
                self._est_varint(r, list(self.bhist))
        t = self.m.st_case[self._case_ctx()]
        cum, freq, tot = t.find_encode(flag)
        self._est_bits(acc, cum, freq, tot)
        return acc[0]

    def _est_raw(self, data):
        """Оценка сырого литерала; одиночный байт кодируется без st_len."""
        if len(data) == 1:
            return self._est_byte(data[0], list(self.bhist))
        acc = [0]
        hist = list(self.bhist)
        i = 0
        while i < len(data):
            k = len(data) - i
            if k > 63:
                chunk, s = data[i:i + 63], 62
            elif k == 63:
                chunk, s = data[i:i + 63], 63
            else:
                chunk, s = data[i:i + k], k - 1
            t = self.m.st_len
            cum, freq, tot = t.find_encode(s)
            self._est_bits(acc, cum, freq, tot)
            for b in chunk:
                acc[0] += self._est_byte(b, hist)
                hist.append(b)
            i += len(chunk)
        return acc[0]

    def _est_punct(self, pidx):
        """Оценка структурного punct-пути (A/C при попадании, иначе O0)."""
        acc = [0]
        ft = self._ctx_ac()
        hit = False
        if ft is not None:
            hit = self._est_ft(acc, ft, pidx, None)
        if not hit:
            o0 = self.m.st_o0
            cum, freq, tot = o0.find_encode(pidx, None)
            self._est_bits(acc, cum, freq, tot)
        return acc[0]

    # ------------------------------------------------------- word chain
    def _o0_ctx(self):
        """Язык предыдущего слова: 1 кириллица / 2 латиница / 0 прочее."""
        if self.w1 is None:
            return 0
        return self.m.word_lang[self.w1 - self.m.P]

    def _word_tables(self):
        """Упорядоченный список (ft, tag) попыток для слова."""
        m = self.m
        tabs = []
        pair = None
        if self.w1 is not None and self.w2 is not None:
            pair = (self.w2 << PAIR_SHIFT) | self.w1
            mft = self.match.get(pair)
            if mft is not None:
                tabs.append((mft, "match"))
            ft = m.b2.get(pair)
            if ft is not None:
                tabs.append((ft, "o2"))
        prev = self.prev
        if prev is not None and prev[0] == "punct":
            p = m.puncts[prev[1]]
            if p == " ":
                if self.w1 in m.b1:
                    tabs.append((m.b1[self.w1], "b1"))
            else:
                ft = m.C.get(prev[1])
                if ft is not None:
                    tabs.append((ft, "c"))
        if self.w1 is not None:
            li = m.lemma_ids[self.w1 - m.P]
            ft = m.lem.get(li)
            if ft is not None:
                tabs.append((ft, "lem"))
        return tabs

    def _lzp_predicted(self):
        """Предсказанный sym по паре (w2,w1) из истории, либо None."""
        if self.w1 is None or self.w2 is None:
            return None
        pair = (self.w2 << PAIR_SHIFT) | self.w1
        j = self.lzp_pos.get(pair)
        if j is None or j >= len(self.wlist):
            return None
        return self.wlist[j]

    def _lzp_register(self, sym):
        """Регистрация текущей позиции в LZP-памяти (вызывается для каждого слова)."""
        if self.w1 is not None and self.w2 is not None:
            pair = (self.w2 << PAIR_SHIFT) | self.w1
            self.lzp_pos[pair] = len(self.wlist)
        self.wlist.append(sym)

    def _enc_lzp_hit(self, enc):
        t = self.lzp_flag
        cum, freq, tot = t.find_encode(1)
        enc.encode(cum, freq, tot)
        t.adapt(1, insert=False)

    def _enc_lzp_miss(self, enc):
        t = self.lzp_flag
        cum, freq, tot = t.find_encode(0)
        enc.encode(cum, freq, tot)
        t.adapt(0, insert=False)

    def _dec_lzp(self, dec):
        t = self.lzp_flag
        cum = dec.start(t.total())
        s, lo, freq = t.decode_sym(cum)
        dec.finish(lo, freq)
        t.adapt(s, insert=False)
        return s == 1

    def _after_word(self, sym):
        """Обновление match-памяти и состояния (общее для enc/dec)."""
        if self.w1 is not None and self.w2 is not None:
            pair = (self.w2 << PAIR_SHIFT) | self.w1
            mft = self.match.get(pair)
            if mft is None:
                self.match[pair] = FT([sym], [W_MATCH], ESC_MATCH)
            elif mft.bs and mft.bs[0] == sym:
                mft.adapt(sym, insert=False)
            else:
                self.match[pair] = FT([sym], [W_MATCH], ESC_MATCH)
        self.w2, self.w1 = self.w1, sym
        self.prev2, self.prev = self.prev, ("word", sym)

    # ------------------------------------------------------------ encode
    def encode(self, text):
        """Конверт: токенный (0x02) или байтовый (0x03/0x04/0x05 — глубина цепочки
        o3+o2+o0 / o2+o0 / o0). Кодируем всеми, отдаём меньший; режим — по тегу."""
        best_tag, best_buf = 2, self._encode_tokens(text)
        for mode in (0, 1, 2):
            buf = self._encode_bytes(text, mode)
            if len(buf) < len(best_buf):
                best_tag, best_buf = 3 + mode, buf
        return bytes([best_tag]) + best_buf

    def _encode_tokens(self, text):
        m = self.m
        self._fresh()
        self.byte_mode = None
        enc = RangeEncoder()
        toks = tokenize(text)
        punct_idx = {p: i for i, p in enumerate(m.puncts)}
        for kind, tok in toks:
            if kind == "word":
                low = tok.lower()
                r = m.rank.get(low)
                flag = get_case(tok)
                ok = r is not None and flag != 3 and apply_case(m.words[r], flag) == tok
                if ok:
                    self._enc_kind(enc, 0)
                    sym = m.P + r
                    pred = self._lzp_predicted()
                    hit = pred is not None and sym == pred
                    if pred is not None:
                        if hit:
                            self._enc_lzp_hit(enc)
                        else:
                            self._enc_lzp_miss(enc)
                    if hit:
                        self._enc_case(enc, flag, self._case_ctx())
                        self._lzp_register(sym)
                        self._after_word(sym)
                        self.bhist.extend(tok.encode("utf-8"))
                        continue
                    # выбор: словарная цепочка или сырые байты
                    tb = tok.encode("utf-8")
                    use_chain = self._est_chain(sym, flag, pred, r) <= self._est_raw(tb)
                    if not use_chain:
                        # сырые байты; состояние синхронизируется (декодер восстановит sym)
                        if len(tb) == 1:
                            self._enc_raw_flag(enc, 0)
                            self._enc_byte(enc, tb[0])
                        else:
                            self._enc_raw_flag(enc, 2)
                            self._enc_lit(enc, tb)
                        self._lzp_register(sym)
                        self._after_word(sym)
                        continue
                    self._enc_raw_flag(enc, 1)
                    excl = set()
                    if pred is not None:
                        excl.add(pred)      # LZP-промах: предсказанное исключается
                    escaped = []
                    done = False
                    for ft, tag in self._word_tables():
                        if ft.in_table(sym, excl):
                            cum, freq, tot = ft.find_encode(sym, excl)
                            enc.encode(cum, freq, tot)
                            ft.adapt(sym, insert=False)
                            self.m.touched.add(ft)
                            done = True
                            break
                        cum, freq, tot = ft.find_encode(None, excl)
                        enc.encode(cum, freq, tot)
                        escaped.append(ft)
                        self.m.touched.add(ft)
                        excl.update(ft.bs)
                        excl.update(ft.delta.keys())
                    if not done:
                        o0 = m.st_o0
                        ctx = self._o0_ctx()
                        if sym in o0.bset and sym not in excl:
                            cum, freq, tot = o0.find_encode(sym, excl, ctx)
                            enc.encode(cum, freq, tot)
                            o0.adapt(sym, insert=False)
                        else:
                            cum, freq, tot = o0.find_encode(ESC_RANK, excl, ctx)
                            enc.encode(cum, freq, tot)
                            o0.adapt(ESC_RANK, insert=False)
                            self._enc_varint(enc, r)
                        self.m.touched.add(o0)
                    for ft in escaped:
                        ft.adapt(None, insert=True, insert_sym=sym)
                    self._enc_case(enc, flag, self._case_ctx())
                    self._lzp_register(sym)
                    self._after_word(sym)
                    self.bhist.extend(tok.encode("utf-8"))
                else:
                    tb = tok.encode("utf-8")
                    if len(tb) == 1:
                        self._enc_kind(enc, 2)
                        self._enc_byte(enc, tb[0])
                    else:
                        self._enc_kind(enc, 4)
                        self._enc_lit(enc, tb)
                    self.prev2, self.prev = self.prev, ("lit",)
            elif kind == "lit":
                pidx = punct_idx.get(tok)
                if pidx is not None:
                    self._enc_kind(enc, 1)
                    tb = tok.encode("utf-8")
                    use_st = self._est_punct(pidx) <= self._est_raw(tb)
                    if not use_st:
                        if len(tb) == 1:
                            self._enc_raw_flag(enc, 0)
                            self._enc_byte(enc, tb[0])
                        else:
                            self._enc_raw_flag(enc, 2)
                            self._enc_lit(enc, tb)
                        self.prev2, self.prev = self.prev, ("punct", pidx)
                        continue
                    self._enc_raw_flag(enc, 1)
                    done = False
                    ft = self._ctx_ac()
                    if ft is not None:
                        if ft.in_table(pidx):
                            cum, freq, tot = ft.find_encode(pidx)
                            enc.encode(cum, freq, tot)
                            ft.adapt(pidx, insert=False)
                            done = True
                        else:
                            cum, freq, tot = ft.find_encode(None)
                            enc.encode(cum, freq, tot)
                            ft.adapt(None, insert=False)
                        self.m.touched.add(ft)
                    if not done:
                        o0 = m.st_o0
                        cum, freq, tot = o0.find_encode(pidx)
                        enc.encode(cum, freq, tot)
                        o0.adapt(pidx, insert=False)
                        self.m.touched.add(o0)
                    self.prev2, self.prev = self.prev, ("punct", pidx)
                    self.bhist.extend(tok.encode("utf-8"))
                else:
                    tb = tok.encode("utf-8")
                    if len(tb) == 1:
                        self._enc_kind(enc, 2)
                        self._enc_byte(enc, tb[0])
                    else:
                        self._enc_kind(enc, 4)
                        self._enc_lit(enc, tb)
                    self.prev2, self.prev = self.prev, ("lit",)
        self._enc_kind(enc, 3)      # EOS
        return enc.flush()

    def _encode_bytes(self, text, mode):
        """Чисто байтовый режим глубины mode: varint-длина + поток байт."""
        self._fresh()
        self.byte_mode = mode
        enc = RangeEncoder()
        data = text.encode("utf-8")
        self._enc_varint(enc, len(data))
        for b in data:
            self._enc_byte(enc, b)
        return enc.flush()

    def _enc_raw_flag(self, enc, v):
        t = self.raw_t
        cum, freq, tot = t.find_encode(v)
        enc.encode(cum, freq, tot)
        t.adapt(v, insert=False)

    def _enc_kind(self, enc, k):
        t = self._kind_ft()
        cum, freq, tot = t.find_encode(k)
        enc.encode(cum, freq, tot)
        t.adapt(k, insert=False)
        self.last_kind = k

    def _kind_ft(self):
        t = self.kindc.get(self.last_kind)
        if t is None:
            t = FT([0, 1, 2, 3, 4], [650, 260, 90, 24, 40], 0)
            self.kindc[self.last_kind] = t
        return t

    # ------------------------------------------------------------ decode
    def decode(self, data):
        if 3 <= data[0] <= 5:
            return self._decode_bytes(data[1:], data[0] - 3)
        assert data[0] == _FMT_TAG, "bad header"
        m = self.m
        self._fresh()
        self.byte_mode = None
        dec = RangeDecoder(data[1:])
        out = []
        punct_idx = {p: i for i, p in enumerate(m.puncts)}
        while True:
            if dec.p > len(dec.d) + 16:
                raise ValueError("stream exhausted (рассинхронизация)")
            k = self._dec_kind(dec)
            if k == 3:
                break
            if k == 0:
                pred = self._lzp_predicted()
                excl = set()
                escaped = []
                sym = None
                hit = False
                if pred is not None:
                    hit = self._dec_lzp(dec)
                    if hit:
                        sym = pred           # попадание LZP: цепочка не нужна
                        flag = self._dec_case(dec, self._case_ctx())
                        w = apply_case(m.words[sym - m.P], flag)
                        out.append(w)
                        self.bhist.extend(w.encode("utf-8"))
                        self._lzp_register(sym)
                        self._after_word(sym)
                        continue
                    excl.add(pred)
                v = self._dec_raw_sym(dec)
                if v != 1:
                    if v == 0:                     # одиночный байт
                        s = bytes([self._dec_byte(dec)]).decode("utf-8")
                    else:                          # прогон (v == 2)
                        s = self._dec_lit(dec).decode("utf-8")
                    out.append(s)
                    # синхронизация состояния: декодер восстановит sym из словаря
                    r2 = m.rank.get(s.lower())
                    if r2 is not None:
                        sym2 = m.P + r2
                        self._lzp_register(sym2)
                        self._after_word(sym2)
                    continue
                for ft, tag in self._word_tables():
                    cum = dec.start(ft.total_excl(excl))
                    s, lo, freq = ft.decode_sym(cum, excl)
                    dec.finish(lo, freq)
                    self.m.touched.add(ft)
                    if s is not None:
                        ft.adapt(s, insert=False)
                        sym = s
                        break
                    escaped.append(ft)
                    excl.update(ft.bs)
                    excl.update(ft.delta.keys())
                r = None
                if sym is None:
                    o0 = m.st_o0
                    ctx = self._o0_ctx()
                    cum = dec.start(o0.total_excl(excl, ctx))
                    s, lo, freq = o0.decode_sym(cum, excl, ctx)
                    dec.finish(lo, freq)
                    self.m.touched.add(o0)
                    if s == ESC_RANK:
                        o0.adapt(ESC_RANK, insert=False)
                        r = self._dec_varint(dec)
                        sym = m.P + r
                    else:
                        o0.adapt(s, insert=False)
                        sym = s
                for ft in escaped:
                    ft.adapt(None, insert=True, insert_sym=sym)
                self._lzp_register(sym)
                if r is not None:
                    flag = self._dec_case(dec, self._case_ctx())
                    w = apply_case(m.words[r], flag)
                    out.append(w)
                    self.bhist.extend(w.encode("utf-8"))
                else:
                    w = m.words[sym - m.P]
                    flag = self._dec_case(dec, self._case_ctx())
                    w = apply_case(w, flag)
                    out.append(w)
                    self.bhist.extend(w.encode("utf-8"))
                self._after_word(sym)
            elif k == 1:
                v = self._dec_raw_sym(dec)
                if v != 1:
                    if v == 0:
                        s = bytes([self._dec_byte(dec)]).decode("utf-8")
                    else:
                        s = self._dec_lit(dec).decode("utf-8")
                    out.append(s)
                    pidx2 = punct_idx.get(s)
                    if pidx2 is not None:
                        self.prev2, self.prev = self.prev, ("punct", pidx2)
                    else:
                        self.prev2, self.prev = self.prev, ("lit",)
                    continue
                ft = self._ctx_ac()
                sym = None
                if ft is not None:
                    cum = dec.start(ft.total())
                    s, lo, freq = ft.decode_sym(cum)
                    dec.finish(lo, freq)
                    self.m.touched.add(ft)
                    if s is not None:
                        ft.adapt(s, insert=False)
                        sym = s
                    else:
                        ft.adapt(None, insert=False)
                if sym is None:
                    o0 = m.st_o0
                    cum = dec.start(o0.total())
                    s, lo, freq = o0.decode_sym(cum)
                    dec.finish(lo, freq)
                    o0.adapt(s, insert=False)
                    self.m.touched.add(o0)
                    sym = s
                out.append(m.puncts[sym])
                self.prev2, self.prev = self.prev, ("punct", sym)
                self.bhist.extend(m.puncts[sym].encode("utf-8"))
            elif k == 2:
                b = self._dec_byte(dec)          # одиночный байт литерала (ASCII)
                out.append(bytes([b]).decode("utf-8"))
                self.prev2, self.prev = self.prev, ("lit",)
            else:
                lit = self._dec_lit(dec)
                out.append(lit.decode("utf-8"))
                self.prev2, self.prev = self.prev, ("lit",)
        return "".join(out)

    def _decode_bytes(self, data, mode):
        """Чисто байтовый режим глубины mode: varint-длина + поток байт."""
        self._fresh()
        self.byte_mode = mode
        dec = RangeDecoder(data)
        n = self._dec_varint(dec)
        out = bytearray()
        for _ in range(n):
            out.append(self._dec_byte(dec))
        return out.decode("utf-8")

    def _dec_raw_sym(self, dec):
        """Значение флага пути: 0=сырой 1 байт, 1=структурный, 2=сырой прогон."""
        t = self.raw_t
        cum = dec.start(t.total())
        s, lo, freq = t.decode_sym(cum)
        dec.finish(lo, freq)
        t.adapt(s, insert=False)
        return s

    def _dec_raw_flag(self, dec):
        t = self.raw_t
        cum = dec.start(t.total())
        s, lo, freq = t.decode_sym(cum)
        dec.finish(lo, freq)
        t.adapt(s, insert=False)
        return s == 1

    def _dec_kind(self, dec):
        t = self._kind_ft()
        cum = dec.start(t.total())
        s, lo, freq = t.decode_sym(cum)
        dec.finish(lo, freq)
        t.adapt(s, insert=False)
        self.last_kind = s
        return s


# ------------------------------------------------------------- API
_MODEL = None


def get_model():
    global _MODEL
    if _MODEL is None:
        _MODEL = Model(MODEL_PATH)
    return _MODEL


def compress(text):
    return SZGC2(get_model()).encode(text)


def decompress(data):
    return SZGC2(get_model()).decode(data)


# -------------------------------------------------------------
def selftest():
    ok = 0
    texts = []
    for p in sorted(glob.glob(os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                            "data", "corpus", "wiki_00.txt"))):
        with open(p, encoding="utf-8") as f:
            first = f.read(6000)
            f.seek(0, 2)
            size = f.tell()
            f.seek(max(0, size // 2))
            mid = f.read(6000)
        texts += [first, mid]
    texts += [
        "Привет! Как дела?",
        "Спасибо большое, выручил!",
        "Не забудь купить хлеб и молоко.",
        "Ок, до связи!",
        "",
        " ",
        "Я",
        "ЁЖик и ёлка, APPLE iPhone 15 Про \U0001f600",
        "Числа: 12345 и 67890, дата 2026-10-26.",
        "СМЕШАННЫЙ ЁЖ и Крош",
        "ЦБ России повысил ключевую ставку до двадцати одного процента годовых.",
        "Все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива по-своему.",
    ]
    rnd = random.Random(20261026)
    alphabets = [
        "абвгдеёжзийклмнопрстуфхцчшщъыьэюя ",
        "abc XYZ .,!?\n",
        "😀😃🌍中文日本語",
        "ёЁ ёлка ЁЛКА",
        "\x00\x01\x02\xff",
    ]
    for _ in range(200):
        a = rnd.choice(alphabets)
        nn = rnd.randint(0, 120)
        texts.append("".join(rnd.choice(a) for _ in range(nn)))
    fails = []
    t0 = time.time()
    for i, t in enumerate(texts):
        try:
            c = compress(t)
            d = decompress(c)
        except Exception as e:
            fails.append((i, "EXC %r" % e, t[:60]))
            continue
        if d != t:
            fails.append((i, len(t), t[:60]))
        else:
            ok += 1
        c2 = compress(t)
        if c2 != c:
            fails.append((i, -1, "NONDET " + t[:40]))
        if i % 40 == 0:
            print("  ... %d/%d (%.1fs)" % (i, len(texts), time.time() - t0), flush=True)
    print("round-trip: %d/%d OK (%.1fs)" % (ok, len(texts), time.time() - t0))
    if fails:
        for f in fails[:10]:
            print("FAIL:", f)
        sys.exit(1)
    print("детерминизм кодировщика: OK")
    print("ALL TESTS PASSED")


if __name__ == "__main__":
    selftest()
'''


In [ ]:
# ---------- 4. Верификация: round-trip, бенчмарк ----------
import os, time, gzip, bz2, lzma, random
import brotli, zstandard

with open('/content/szgc/szgc_codec.py', 'w') as f:
    f.write(CODEC_SRC)
CODEC_SRC = None

import shutil
shutil.copy('/content/graph-model.bin', '/content/szgc/graph-model.bin')

import szgc_codec as S
S.get_model()  # загрузка /content/szgc/graph-model.bin

texts = [
    "Привет! Как дела?",
    "Не забудь купить хлеб и молоко.",
    "Сегодня мы пошли в кино. Завтра мы пошли в театр.",
    "The quick brown fox jumps over the lazy dog. Pack my box with five dozen liquor jugs.",
    "ЦБ России повысил ключевую ставку до двадцати одного процента годовых.",
]
# held-out куски из корпуса (не участвуют в подстройке весов контекстов так же сильно)
import glob
files = sorted(glob.glob('/content/data/corpus/*.txt'))
if files:
    with open(files[-1], encoding='utf-8', errors='replace') as f:
        texts.append(f.read(4000))

rnd = random.Random(42)
alphabets = ["абвгдеёжзийклмнопрстуфхцчшщъыьэюя ", "abc XYZ .,!?\n", "😀😃🌍中文日本語"]
for _ in range(20):
    a = rnd.choice(alphabets)
    texts.append("".join(rnd.choice(a) for _ in range(rnd.randint(0, 120))))

fails = 0
for i, t in enumerate(texts):
    c = S.compress(t)
    if S.decompress(c) != t:
        fails += 1
        print('ROUND-TRIP FAIL', i)
    if S.compress(t) != c:
        fails += 1
        print('NONDET FAIL', i)
print('round-trip + детерминизм: %d/%d OK' % (len(texts) - fails, len(texts)))
assert fails == 0

def bench_row(t):
    b = t.encode('utf-8')
    zstd = zstandard.ZstdCompressor(level=19)
    row = {'raw': len(b), 'szgc': len(S.compress(t)),
           'gzip9': len(gzip.compress(b, 9)), 'bz2': len(bz2.compress(b, 9)),
           'xz9e': len(lzma.compress(b, preset=9 | lzma.PRESET_EXTREME)),
           'brotli11': len(brotli.compress(b, quality=11))}
    try:
        row['zstd19'] = len(zstd.compress(b))
    except Exception:
        row['zstd19'] = -1
    return row

print('%-8s %6s %6s %6s %6s %6s %6s %6s' % ('text', 'raw', 'SZGC', 'gzip9', 'bz2', 'xz9e', 'br11', 'zstd19'))
for i, t in enumerate(texts):
    r = bench_row(t)
    print('%-8s %6d %6d %6d %6d %6d %6d %6d' %
          ('text%d' % i, r['raw'], r['szgc'], r['gzip9'], r['bz2'], r['xz9e'], r['brotli11'], r['zstd19']))
print('верификация завершена')


In [ ]:
# ---------- 5. Отчёт и выгрузка модели ----------
import json
stats = json.load(open('/content/model-stats.json'))
print(json.dumps(stats, ensure_ascii=False, indent=1)[:1200])
from google.colab import files
files.download('/content/graph-model.bin')
files.download('/content/model-stats.json')
